In [ ]:
# Optional: confirm that Tesseract is available on this system.
# This is useful when OCR is needed for scanned pages in the PDF.
# import subprocess
# result = subprocess.run(["which", "tesseract"], capture_output=True, text=True)
# print(result.stdout)  # shows the actual path


# Project workflow

This notebook demonstrates the end-to-end pipeline for turning a PDF into searchable embeddings:

1. Extract semantic elements from the PDF with `unstructured`
2. Normalize the output into a stable schema
3. Chunk the document by headings and section boundaries
4. Generate vector embeddings
5. Store the chunks in ChromaDB
6. Run a semantic search query

The sample input lives in `Training_Data/` and the vector store is written to `./chroma_db`.


# 1. Basic PDF Preprocessing with Unstructured

This step extracts structured semantic elements from the PDF, including headings, paragraphs, lists, and other document blocks.

The `hi_res` strategy is used here because it performs better on formatted documents and helps preserve layout-aware content for downstream retrieval.


In [ ]:
import os

# Ensure Homebrew's Tesseract binary is available for OCR on macOS.
# This is optional if OCR is not required for the current PDF.
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]


In [ ]:
from unstructured.partition.pdf import partition_pdf

# Extract semantic elements from the sample PDF.
# `hi_res` is selected for cleaner layout-aware parsing and better OCR fallback.
elements = partition_pdf(
    filename="Training_Data/AML_NOTES_UNIT_1_2_3_4_5_merged.pdf",
    strategy="hi_res",
    languages=["eng"],
    include_metadata=True,
    infer_table_structure=True,
    extract_images_in_pdf=False,
)


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 10098.47it/s]


In [5]:
print(f"Total elements: {len(elements)}")

Total elements: 853


In [6]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 80)

<class 'unstructured.documents.elements.Title'>
Unsupervised Learning :
<class 'unstructured.documents.elements.Title'>
What is Unsupervised Learning?
<class 'unstructured.documents.elements.NarrativeText'>
As the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:
<class 'unstructured.documents.elements.NarrativeText'>
Unsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.
<class 'unstructured.documents.elements.NarrativeText'>
Unsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output dat

# 2. Normalize the Output

`unstructured` returns a mix of element types such as `Title`, `NarrativeText`, `Table`, and `ListItem`. To make the pipeline predictable, we normalize the output into a single schema with a consistent set of fields.

This is useful when you later chunk, embed, and store content because every record follows the same structure.


In [ ]:
normalized_docs = []

# Convert heterogeneous `unstructured` elements into one consistent schema.
# This makes downstream processing simpler and reusable across files.
for idx, el in enumerate(elements):
    doc = {
        "id": f"doc_{idx}",
        "type": el.category,
        "text": el.text,
        "page_number": getattr(el.metadata, "page_number", None),
        "filename": getattr(el.metadata, "filename", None),
        "languages": getattr(el.metadata, "languages", None),
        "coordinates": str(getattr(el.metadata, "coordinates", None)),
    }
    normalized_docs.append(doc)


In [9]:
print(normalized_docs[0])

{'id': 'doc_0', 'type': 'Title', 'text': 'Unsupervised Learning :', 'page_number': 1, 'filename': 'AML_NOTES_UNIT_1_2_3_4_5_merged.pdf', 'languages': ['eng'], 'coordinates': 'CoordinatesMetadata(points=((np.float64(455.00001093749995), np.float64(350.380126953125)), (np.float64(455.00001093749995), np.float64(433.4489440917969)), (np.float64(1287.67214583046), np.float64(433.4489440917969)), (np.float64(1287.67214583046), np.float64(350.380126953125))), system=<unstructured.documents.coordinates.PixelSpace object at 0x3350da1b0>)', 'source': None}


# 3. Chunk the Document Properly

Chunking happens after extraction so the model receives semantically meaningful segments rather than large, unstructured blobs of text.

Using `chunk_by_title()` helps keep sections together, preserves headings, and reduces the risk of mixing unrelated topics in a single chunk.


Why chunk_by_title() is better:

* respects headings
* preserves sections
* prevents chunk mixing across topics
* ideal for PDFs

In [ ]:
from unstructured.chunking.title import chunk_by_title

# Split the extracted content into section-aware chunks.
# This keeps headings and related paragraphs together, which improves retrieval quality.
chunks = chunk_by_title(
    elements,
    max_characters=1200,
    new_after_n_chars=1000,
    combine_text_under_n_chars=200,
)


In [11]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 140


In [12]:
for chunk in chunks[:3]:
    print(chunk.text)
    print("=" * 80)

Unsupervised Learning :

What is Unsupervised Learning?

As the name suggests, unsupervised learning is a machine learning technique in which models are not supervised using training dataset. Instead, models itself find the hidden patterns and insights from the given data. It can be compared to learning which takes place in the human brain while learning new things. It can be defined as:

Unsupervised learning is a type of machine learning in which models are trained using unlabelled dataset and are allowed to act on that data without any supervision.

Unsupervised learning cannot be directly applied to a regression or classification problem because unlike supervised learning, we have the input data but no corresponding output data. The goal of unsupervised learning is to find the underlying structure of dataset, group that data according to similarities, and represent that dataset in a compressed format.
Example: Suppose the unsupervised learning algorithm is given an input dataset co

# 4. Create Embeddings

This step converts each chunk into a dense numeric representation so the text can be searched semantically.

`all-MiniLM-L6-v2` is a lightweight sentence-transformer model that works well for local experimentation and fast retrieval pipelines.


In [ ]:
from sentence_transformers import SentenceTransformer

# Use a compact sentence-transformer model for local embedding generation.
# It is a good default for experimentation and retrieval pipelines.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [chunk.text for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)


Batches: 100%|██████████| 5/5 [00:00<00:00,  8.57it/s]


# Bonus Steps

# 5. Insert Embeddings into ChromaDB

This section writes the chunk embeddings and their metadata into a persistent ChromaDB collection so the data can be reused across sessions.


In [ ]:
import chromadb

# Connect to a persistent ChromaDB directory so embeddings remain available across runs.
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="single_pdf_collection"
)


In [ ]:
for i, chunk in enumerate(chunks):
    # Store each chunk along with its embedding and basic metadata.
    collection.add(
        ids=[f"chunk_{i}"],
        documents=[chunk.text],
        embeddings=[embeddings[i].tolist()],
        metadatas=[{
            "source": "sample.pdf",
            "chunk_id": i,
        }]
    )

print("Inserted into ChromaDB")


Inserted into ChromaDB


# 6. Query in ChromaDB

The final step embeds the user query and searches the vector store for the most semantically similar chunks.

This makes it possible to retrieve context that matches the intent of the query, even if the wording differs from the source document.


In [ ]:
query = "What is k means clustering?"

# Embed the search query and retrieve the closest matching chunks from ChromaDB.
query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print(results["documents"])
